# Chapter 3 - Lab 3: <font color='blue'>CrewAI Multi-Agent Crew</font>

**<font color='purple'>Goal</font>**:
In this lab, you will build a **financial analysis agent using CrewAI** that compares the P/E (Price/Earnings) ratios of two companies — Apple (AAPL) and JPMorgan (JPM) — and produces a short investment memo.

Where LangChain and ADK gave you a single agent with tools, CrewAI takes a different stance: **model the problem as a crew of specialised agents collaborating on a sequence of tasks**. For our P/E comparison you will define four roles — a Manager, a Data Analyst, a Quant, and an Analyst — and three sequential tasks that pass results between them.

This is the *first multi-agent* lab of the chapter, and it foreshadows the architectural styles you will study in Chapter 7.

This is the same reference task used across every framework lab in Chapter 3 — comparing all of them on the *same* task makes the differences in API style, abstractions, and ergonomics easy to spot.

**<font color='purple'>Tech stack</font>**:

* **CrewAI** (`crewai`) — role/task/crew abstraction for collaborative agents.
* **OpenAI** `gpt-4o-mini` (default) — drives each agent's reasoning.
* **Sequential task graph** — tasks pass `context` to one another.

You will need an OpenAI API key with some credits available.

## 1. Install packages

Install the framework and its dependencies.

In [2]:
%pip install -q crewai 'crewai[tools]' pydantic python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.9/198.9 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 841.2/841.2 kB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 105.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19

In [4]:
%pip check

ipython 7.34.0 requires jedi, which is not installed.
google-colab 1.0.0 has requirement requests==2.32.4, but you have requests 2.34.2.
bigframes 2.48.0 has requirement rich<14,>=12.4.4, but you have rich 14.3.4.
google-adk 2.7.1 has requirement opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0.
google-adk 2.7.1 has requirement opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0.


In [5]:
import crewai
import pydantic

print("CrewAI:", crewai.__version__)
print("Pydantic:", pydantic.__version__)

CrewAI: 1.15.21
Pydantic: 2.12.5


## 2. Set up the API key(s)

This lab needs the following key(s):

  * **`OPENAI_API_KEY`** — your OpenAI key

If you are running this notebook in **Google Colab**, add each key in the left vertical menu under the *key* icon, using the names above.

If you are running locally, set the same names as environment variables (or load them from a `.env` file).

In [6]:
import os

try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally — assume env vars are already set (e.g. via .env).
    pass

## 3. Bootstrap the shared task setup

Every framework lab in this chapter shares the same task, tools, finance dataset, and prompts. These are factored out into `common.py`. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [7]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%203/common.py',
        'common.py',
    )

from common import (
    get_stock_data,
    compute_pe,
    finance_data,
    system_message,
    input_message,
)

print('Tools loaded. Reference task:')
print(' ', input_message)

Tools loaded. Reference task:
  Compare Apple (AAPL) and JPMorgan (JPM) on P/E ratios and summarize in a memo.


## 4. Wrap the shared tools as CrewAI tools

CrewAI exposes a `@tool` decorator that adapts plain Python functions for use by agents. We wrap our two shared functions so the Data Analyst and Quant agents can call them.

In [8]:
from crewai.tools import tool

@tool('get_stock_data_tool')
def get_stock_data_crew(ticker: str) -> str:
    """Fetch price and EPS for a ticker (e.g. AAPL, JPM)."""
    data = get_stock_data(ticker)
    return f'{ticker}: price=${data.price}, EPS=${data.eps}'


@tool('compute_pe_tool')
def compute_pe_crew(price: float, eps: float) -> float:
    """Compute the P/E ratio: price / EPS."""
    return compute_pe(price, eps)

## 5. Define the agents (roles)

Each agent in a CrewAI crew is described by a **role**, a **goal**, and a **backstory**. The backstory shapes the agent's persona. Tools are attached to the agents that need them — the Data Analyst fetches data, the Quant computes ratios.

In [9]:
from crewai import Agent, Crew, Task

manager = Agent(
    role='Manager',
    goal='Coordinate analysis',
    backstory='You are a portfolio manager.',
)

data_agent = Agent(
    role='Data Analyst',
    goal='Fetch stock prices and EPS for AAPL and JPM',
    backstory='You retrieve and clean financial data.',
    tools=[get_stock_data_crew],
)

compute_agent = Agent(
    role='Quant',
    goal='Compute P/E ratios from price and EPS',
    backstory='You calculate ratios and prepare metrics.',
    tools=[compute_pe_crew],
)

narrative_agent = Agent(
    role='Analyst',
    goal='Write a comparative investment memo',
    backstory='You explain investment insights in plain English.',
)

## 6. Define the tasks

Tasks form a small **DAG** — `task2` lists `task1` in its `context`, so it receives `task1`'s output. `task3` chains off `task2`. This declarative wiring is CrewAI's way of expressing 'who does what, in what order, with what inputs'.

In [10]:
task1 = Task(
    description='Get AAPL and JPM prices and EPS using get_stock_data_tool.',
    agent=data_agent,
    expected_output='Prices and EPS for AAPL and JPM.',
)
task2 = Task(
    description='Compute P/E ratios for AAPL and JPM using compute_pe_tool.',
    agent=compute_agent,
    context=[task1],
    expected_output='P/E ratios for AAPL and JPM.',
)
task3 = Task(
    description='Summarise the findings in a short investment memo.',
    agent=narrative_agent,
    context=[task2],
    expected_output='Investment memo comparing AAPL and JPM.',
)

## 7. Assemble the crew and run it

Pass the agents and the tasks into a `Crew`, then `kickoff()`. CrewAI handles the execution order, the context propagation, and the LLM calls behind the scenes.

In [12]:
import asyncio

crew = Crew(
    agents=[manager, data_agent, compute_agent, narrative_agent],
    tasks=[task1, task2, task3],
)

async def run_crew():
    result = await crew.kickoff_async()
    print(result)

await run_crew()

Investment Memo: Comparative Analysis of Apple Inc. (AAPL) and JPMorgan Chase & Co. (JPM)

Overview:
This memo provides a comparative investment insight into two prominent stocks: Apple Inc. (AAPL) and JPMorgan Chase & Co. (JPM). The primary focus is on their Price-to-Earnings (P/E) ratios as a valuation metric to assess relative market expectations and investment potential.

Key Findings:

1. P/E Ratio Comparison
- Apple (AAPL): 29.28
- JPMorgan Chase (JPM): 11.79
The P/E ratio indicates how much investors are willing to pay per dollar of earnings. Apple's significantly higher P/E suggests that the market expects stronger future growth or values its earnings more highly relative to JPM.

2. Interpretation
- AAPL’s higher P/E ratio typically reflects investor confidence in its innovation, brand strength, and growth prospects in products and services such as iPhones, wearables, and cloud services.
- JPM’s lower P/E ratio may indicate more stable, cyclical earnings typical of financial i

## 8. Results

You should see the Data Analyst call `get_stock_data_tool` for AAPL and JPM, the Quant call `compute_pe_tool` to compute each ratio, and the Analyst produce a short memo comparing the two — all coordinated by the Manager.

**What to notice about CrewAI specifically:**

* You did not have to write the orchestration loop yourself — the *crew* abstraction encodes the workflow.
* Specialised roles give each LLM call a focused prompt, which can yield higher-quality output than a single generalist agent.
* Trade-off: more LLM calls (≥3 here) means more tokens, latency, and cost. The next chapter on multi-agent systems goes deeper into when this trade is worth it.
* You will see CrewAI's *patterns* (specialist roles, sequential pipelines) again in Chapters 7 and 9, framed as architectural styles rather than a specific framework feature.